# MEDGUARD-CXR Academic V2 — Run All

**Research use only. No clinical use, diagnosis, treatment, or patient-care decisions.**

A full run requires the dataset/model permissions described in `docs/academic_v2_prerequisites.md` and the applicable Colab Secrets. This notebook is resumable: completed real runs require validated `DONE.json` artifacts, while unavailable licensed resources remain `blocked_external_access`. Synthetic smoke proof is never reported as research performance.

In [ ]:
# Runtime preflight: secret values are never read or printed.
import importlib.util, os, platform, shutil, subprocess, sys
from pathlib import Path
SMOKE = os.environ.get('MEDGUARD_NOTEBOOK_SMOKE') == '1'
gpu_name, vram_gb, cuda = 'none', 0.0, False
try:
    import torch
    cuda = torch.cuda.is_available()
    if cuda:
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
except Exception:
    pass
secret_names = ['GITHUB_TOKEN','HF_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY','PHYSIONET_USERNAME','PHYSIONET_PASSWORD']
secret_presence = {name: bool(os.environ.get(name)) for name in secret_names}
print({'python': platform.python_version(), 'gpu': gpu_name, 'vram_gb': round(vram_gb, 1), 'cuda': cuda, 'disk_free_gb': round(shutil.disk_usage(Path.cwd()).free / 2**30, 1), 'git': shutil.which('git') is not None, 'secrets_present': secret_presence, 'smoke': SMOKE})

In [ ]:
# Mount Drive and establish the canonical private workspace.
if not SMOKE and importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    drive.mount('/content/drive')
WORKSPACE = Path(os.environ.get('MEDGUARD_V2_WORKSPACE', '/tmp/medguard-academic-v2-smoke' if SMOKE else '/content/drive/MyDrive/medguard-cxr-academic-v2')).resolve()
for name in ['repo','datasets','dataset_cache','model_cache','runs','restricted_predictions','reports','release','logs','results_v2']:
    (WORKSPACE / name).mkdir(parents=True, exist_ok=True)
print(WORKSPACE)

In [ ]:
# Repository sync; local uncommitted changes are preserved.
REPO = Path.cwd().resolve() if SMOKE else WORKSPACE / 'repo'
repo_url = os.environ.get('MEDGUARD_REPO_URL', 'https://github.com/ColdVI/medguard-cxr.git')
if not SMOKE:
    if not (REPO / '.git').exists():
        subprocess.run(['git', 'clone', repo_url, str(REPO)], check=True)
    else:
        dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO, text=True, capture_output=True, check=True).stdout.strip()
        subprocess.run(['git', 'fetch', '--all', '--prune'], cwd=REPO, check=True)
        if not dirty:
            subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
        else:
            print('Repository has uncommitted changes; pull skipped without deleting them.')
assert (REPO / 'configs/programs/academic_v2_full.yaml').is_file(), 'Academic V2 schema is missing'
os.chdir(REPO)
print(REPO)

In [ ]:
# Separate core/VLM environments; unchanged lock hashes are skipped.
import hashlib
def lock_hash(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
if not SMOKE:
    for env_name, lock in [('.venv-core','requirements/colab-core.lock'),('.venv-vlm','requirements/colab-vlm.lock')]:
        env_path = WORKSPACE / env_name
        marker = env_path / '.medguard-lock-sha256'
        expected = lock_hash(lock)
        if not marker.exists() or marker.read_text().strip() != expected:
            subprocess.run([sys.executable, '-m', 'venv', str(env_path)], check=True)
            subprocess.run([str(env_path / 'bin/pip'), 'install', '-r', lock], check=True)
            marker.write_text(expected)
CORE_PYTHON = sys.executable if SMOKE else str(WORKSPACE / '.venv-core/bin/python')
VLM_PYTHON = sys.executable if SMOKE else str(WORKSPACE / '.venv-vlm/bin/python')
print({'core_python': CORE_PYTHON, 'vlm_python': VLM_PYTHON})

In [ ]:
# Dataset access preflight. Presence never implies a completed experiment.
datasets = {'nih': False, 'chexpert': True, 'mimic': True, 'vindr': True, 'rsna': True}
access_rows = []
for dataset, restricted in datasets.items():
    root = WORKSPACE / 'datasets' / dataset
    present = root.exists() and any(root.iterdir())
    access_rows.append({'dataset': dataset, 'access': present, 'metadata': present, 'images': present, 'labels': present, 'reports': present if dataset == 'mimic' else False, 'boxes': present if dataset in {'vindr','rsna'} else False, 'status': 'pending' if present else 'blocked_external_access', 'required_action': '' if present else 'See docs/academic_v2_prerequisites.md', 'restricted': restricted})
try:
    import pandas as pd
    display(pd.DataFrame(access_rows))
except Exception:
    print(access_rows)

In [ ]:
# Research orchestrator. Synthetic mode validates plumbing only.
profile = 'synthetic' if SMOKE else 'full'
command = [CORE_PYTHON, 'scripts/run_academic_v2.py', '--config', 'configs/programs/academic_v2_full.yaml', '--workspace', str(WORKSPACE), '--profile', profile, '--resume']
subprocess.run(command, check=True)

In [ ]:
# Live/progress snapshot persisted by the orchestrator.
import json
summary_path = WORKSPACE / 'results_v2/orchestrator_summary.json'
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))
print({'gpu_allocated_mb': round(torch.cuda.memory_allocated() / 2**20, 1) if cuda else 0, 'artifact': str(summary_path)})

In [ ]:
# Strict full finalization fails honestly while mandatory runs are incomplete.
finalize = [CORE_PYTHON, 'scripts/finalize_academic_v2.py', '--workspace', str(WORKSPACE)]
finalize += ['--smoke'] if SMOKE else ['--strict']
subprocess.run(finalize, check=True)

In [ ]:
# Final summary. Missing tables/releases stay visibly absent.
final_status = json.loads((WORKSPACE / 'results_v2/finalization_status.json').read_text())
paths = {name: str(WORKSPACE / 'results_v2' / name) for name in ['classification_leaderboard.csv','cross_dataset_matrix.csv','calibration_comparison.csv','localization_comparison.csv','vlm_comparison.csv']}
print(json.dumps({'finalization': final_status, 'result_paths': paths, 'report': str(WORKSPACE / 'reports/report_academic_v2.pdf'), 'release': str(WORKSPACE / 'release/medguard-cxr-academic-v2-release.zip')}, indent=2))